# Лабораторна робота 4: Побудова Star Schema

Перетворюємо транзакційну OLTP-модель в аналітичну Star Schema.

---
**Мета:** створити DWH з таблицями фактів і вимірів, зробити ETL, виконати аналітичні запити.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect(':memory:')
print('SQLite connected')

---
## Крок 1: Створюємо OLTP-таблиці (source)

Симулюємо транзакційну базу інтернет-магазину.

In [ ]:
conn.executescript('''
    CREATE TABLE source_products (
        product_id INT PRIMARY KEY,
        product_name TEXT,
        category TEXT,
        subcategory TEXT,
        unit_price REAL
    );

    CREATE TABLE source_customers (
        customer_id INT PRIMARY KEY,
        first_name TEXT,
        last_name TEXT,
        email TEXT,
        city TEXT,
        country TEXT,
        signup_date TEXT
    );

    CREATE TABLE source_orders (
        order_id INT PRIMARY KEY,
        customer_id INT,
        order_date TEXT,
        status TEXT
    );

    CREATE TABLE source_order_items (
        item_id INT PRIMARY KEY,
        order_id INT,
        product_id INT,
        quantity INT,
        unit_price REAL  -- may differ from current product price
    );
''')
print('Source tables created')

In [ ]:
# Заповнюємо source дані
np.random.seed(42)

# Products
products = []
for i in range(1, 21):
    cat = np.random.choice(['Electronics', 'Clothing', 'Books', 'Food'])
    sub = {'Electronics': 'Gadgets', 'Clothing': 'Apparel', 'Books': 'Literature', 'Food': 'Groceries'}[cat]
    products.append((i, f'Product_{i}', cat, sub, round(np.random.uniform(10, 500), 2)))

conn.executemany('INSERT INTO source_products VALUES (?,?,?,?,?)', products)

# Customers
customers = []
for i in range(1, 51):
    customers.append((i, f'First_{i}', f'Last_{i}', f'user{i}@mail.com',
                     np.random.choice(['Kyiv', 'Lviv', 'Kharkiv', 'Odesa', 'Dnipro']),
                     'Ukraine',
                     f'202{np.random.randint(0,4)}-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}'))
conn.executemany('INSERT INTO source_customers VALUES (?,?,?,?,?,?,?)', customers)

# Orders (1000 orders)
for oid in range(1, 1001):
    cid = np.random.randint(1, 51)
    date = datetime(2022, 1, 1) + timedelta(days=int(np.random.randint(0, 1095)))
    status = np.random.choice(['completed', 'completed', 'completed', 'cancelled', 'pending'], p=[0.6, 0.2, 0.1, 0.05, 0.05])
    conn.execute('INSERT INTO source_orders VALUES (?,?,?,?)', (oid, cid, date.strftime('%Y-%m-%d'), status))
    
    # 1-5 items per order
    for _ in range(np.random.randint(1, 6)):
        pid = np.random.randint(1, 21)
        qty = np.random.randint(1, 5)
        price = round(np.random.uniform(10, 500), 2)
        conn.execute('INSERT INTO source_order_items VALUES (?,?,?,?,?)',
                     (oid * 100 + _, oid, pid, qty, price))

print('Source data loaded')
for t in ['source_products', 'source_customers', 'source_orders', 'source_order_items']:
    cnt = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {cnt} rows')

---
## Крок 2: Створюємо Star Schema (DWH)

Dimension Tables спочатку, Fact — потім.

In [ ]:
conn.executescript('''
    CREATE TABLE DimDate (
        date_key INT PRIMARY KEY,
        full_date DATE,
        year INT,
        quarter INT,
        month INT,
        month_name TEXT,
        day INT,
        day_of_week INT,
        day_name TEXT,
        is_weekend BOOLEAN
    );

    CREATE TABLE DimProduct (
        product_key INT PRIMARY KEY,
        product_id INT,
        product_name TEXT,
        category TEXT,
        subcategory TEXT
    );

    CREATE TABLE DimCustomer (
        customer_key INT PRIMARY KEY,
        customer_id INT,
        full_name TEXT,
        email TEXT,
        city TEXT,
        country TEXT
    );

    CREATE TABLE FactSales (
        sales_key INTEGER PRIMARY KEY AUTOINCREMENT,
        date_key INT REFERENCES DimDate(date_key),
        product_key INT REFERENCES DimProduct(product_key),
        customer_key INT REFERENCES DimCustomer(customer_key),
        order_id INT,
        quantity INT,
        unit_price REAL,
        total_amount REAL
    );
''')
print('Star Schema tables created')

---
## Крок 3: ETL — Заповнюємо Dimension Tables

In [ ]:
# 3.1 DimDate — заповнюємо всі дати з діапазону orders
min_date = conn.execute('SELECT MIN(order_date) FROM source_orders').fetchone()[0]
max_date = conn.execute('SELECT MAX(order_date) FROM source_orders').fetchone()[0]

import datetime as dt
start = dt.datetime.strptime(min_date, '%Y-%m-%d')
end = dt.datetime.strptime(max_date, '%Y-%m-%d')

dates = []
while start <= end:
    key = int(start.strftime('%Y%m%d'))
    dates.append((key, start.strftime('%Y-%m-%d'), start.year, (start.month - 1)//3 + 1,
                 start.month, start.strftime('%B'), start.day,
                 start.weekday() + 1, start.strftime('%A'),
                 1 if start.weekday() >= 5 else 0))
    start += dt.timedelta(days=1)

conn.executemany('INSERT OR IGNORE INTO DimDate VALUES (?,?,?,?,?,?,?,?,?,?)', dates)
print(f'DimDate: {len(dates)} rows')

In [ ]:
# 3.2 DimProduct — ETL з source_products
conn.execute('''
    INSERT INTO DimProduct (product_key, product_id, product_name, category, subcategory)
    SELECT row_number() OVER (ORDER BY product_id), product_id, product_name, category, subcategory
    FROM source_products
''')
print(f'DimProduct: {conn.execute("SELECT COUNT(*) FROM DimProduct").fetchone()[0]} rows')
display(pd.read_sql('SELECT * FROM DimProduct LIMIT 5', conn))

In [ ]:
# 3.3 DimCustomer — ETL з source_customers
conn.execute('''
    INSERT INTO DimCustomer (customer_key, customer_id, full_name, email, city, country)
    SELECT row_number() OVER (ORDER BY customer_id), customer_id,
           first_name || ' ' || last_name, email, city, country
    FROM source_customers
''')
print(f'DimCustomer: {conn.execute("SELECT COUNT(*) FROM DimCustomer").fetchone()[0]} rows')

---
## Крок 4: ETL — Заповнюємо FactSales

Об'єднуємо source_orders + source_order_items, мапимо ключі вимірів.

In [ ]:
conn.execute('''
    INSERT INTO FactSales (date_key, product_key, customer_key, order_id, quantity, unit_price, total_amount)
    SELECT 
        CAST(REPLACE(o.order_date, '-', '') AS INT) AS date_key,
        dp.product_key,
        dc.customer_key,
        o.order_id,
        oi.quantity,
        oi.unit_price,
        oi.quantity * oi.unit_price AS total_amount
    FROM source_orders o
    JOIN source_order_items oi ON o.order_id = oi.order_id
    JOIN DimProduct dp ON oi.product_id = dp.product_id
    JOIN DimCustomer dc ON o.customer_id = dc.customer_id
    WHERE o.status = 'completed'
''')
print(f'FactSales: {conn.execute("SELECT COUNT(*) FROM FactSales").fetchone()[0]} rows')
display(pd.read_sql('SELECT * FROM FactSales LIMIT 5', conn))

---
## Крок 5: Аналітика на Star Schema

Мінімум JOIN — максимум швидкості.

In [ ]:
# 5.1 Продажі по категоріях
query1 = '''
    SELECT 
        p.category,
        COUNT(*) AS orders_count,
        SUM(f.quantity) AS units_sold,
        ROUND(SUM(f.total_amount), 2) AS revenue,
        ROUND(AVG(f.total_amount), 2) AS avg_order_value
    FROM FactSales f
    JOIN DimProduct p ON f.product_key = p.product_key
    GROUP BY p.category
    ORDER BY revenue DESC
'''
print('=== Sales by Category ===')
display(pd.read_sql(query1, conn))

In [ ]:
# 5.2 Динаміка по місяцях
query2 = '''
    SELECT 
        d.year,
        d.month,
        d.month_name,
        COUNT(DISTINCT f.customer_key) AS customers,
        COUNT(*) AS orders,
        ROUND(SUM(f.total_amount), 2) AS revenue
    FROM FactSales f
    JOIN DimDate d ON f.date_key = d.date_key
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month
'''
print('=== Monthly Revenue ===')
display(pd.read_sql(query2, conn).head(12))

In [ ]:
# 5.3 Топ-10 клієнтів
query3 = '''
    SELECT 
        c.full_name,
        c.city,
        COUNT(*) AS orders,
        ROUND(SUM(f.total_amount), 2) AS total_spent
    FROM FactSales f
    JOIN DimCustomer c ON f.customer_key = c.customer_key
    GROUP BY c.customer_key
    ORDER BY total_spent DESC
    LIMIT 10
'''
print('=== Top 10 Customers ===')
display(pd.read_sql(query3, conn))

In [ ]:
# 5.4 Порівняння: який день тижня найприбутковіший
query4 = '''
    SELECT 
        d.day_name,
        d.is_weekend,
        COUNT(*) AS orders,
        ROUND(SUM(f.total_amount), 2) AS revenue,
        ROUND(AVG(f.total_amount), 2) AS avg_revenue
    FROM FactSales f
    JOIN DimDate d ON f.date_key = d.date_key
    GROUP BY d.day_name
    ORDER BY revenue DESC
'''
print('=== Revenue by Day of Week ===')
display(pd.read_sql(query4, conn))

---
## Крок 6: Практичні завдання

1. Додайте **DimDate** з колонками `week_number` та `is_holiday`
2. Знайдіть **топ-5 продуктів** за revenue
3. Порахуйте **середній чек (avg order value)** по місяцях
4. Зробіть **pivot**: категорії по колонках, роки по рядках, revenue всередині
5. (Бонус) Додайте другий факт — **FactReturns** (повернення товарів)

In [ ]:
conn.close()
print('Connection closed')